In [1]:
%load_ext autoreload
%autoreload 2

# Calculate Pearson correlations between `intpp` and growth limitation factors for all phytoplankton types

In [2]:
import esm4ppe
import xarray as xr
from dask.diagnostics import ProgressBar
import os
import itertools

In [3]:
from functions import *

In [4]:
bymonth=True
detrend=False
deseasonalize=False
overwrite = False
frequency = 'monthly'

In [5]:
phytotypes = ['diat','diaz','misc','pico']
limitations = ['limfe','limirr','limn','limp']

In [6]:
for (phytotype,limitation) in itertools.product(phytotypes,limitations):
    if (phytotype=='diaz') & (limitation=='limn'):
        # diazotrophs never limited by n, so no diagnostic
        continue
    variable1 = 'intpp'+phytotype
    variable2 = limitation+phytotype
    # get path
    path = esm4ppe.get_path_correlation(variable1,
                                        variable2,
                                        frequency = frequency,
                                        metric='pearson',
                                        bymonth=bymonth,
                                        detrend=detrend,
                                        deseasonalize=deseasonalize)    
    if (os.path.exists(path)) & (overwrite==False):
        print(path+" exists.")
    else:
        # remove old file if it exists
        if os.path.exists(path):
            os.remove(path)
        # load data
        ds = xr.Dataset()
        for variable in [variable1,variable2]:
            es = esm4ppe.esm4ppeObj(variable,frequency)
            ds[variable] = es.add_control().control[variable]
        static = es.static
        # calculate correlation
        cr = calc_pearson(ds[variable1],
                           ds[variable2],
                           bymonth=bymonth,
                           detrend=detrend,
                           deseasonalize=deseasonalize)
        # compute and save
        print("saving to "+path)
        with ProgressBar():
            cr = cr.compute()
        cr.to_netcdf(path,mode='w')

Opening static... static opened.


/home/graemem/.conda/envs/climpred/lib/python3.11/site-packages/xarray/conventions.py:448: SerializationWarning: variable 'thkcello' has multiple fill values {1e+20, 1e+20}, decoding all values to NaN.
  new_vars[k] = decode_cf_variable(
/home/graemem/.conda/envs/climpred/lib/python3.11/site-packages/xarray/conventions.py:448: SerializationWarning: variable 'volcello' has multiple fill values {1e+20, 1e+20}, decoding all values to NaN.
  new_vars[k] = decode_cf_variable(


Control present in zarr store... opening... control opened.
Opening static... static opened.
Control present in zarr store... opening... control opened.


/home/graemem/.conda/envs/climpred/lib/python3.11/site-packages/xarray/conventions.py:448: SerializationWarning: variable 'thkcello' has multiple fill values {1e+20, 1e+20}, decoding all values to NaN.
  new_vars[k] = decode_cf_variable(
/home/graemem/.conda/envs/climpred/lib/python3.11/site-packages/xarray/conventions.py:448: SerializationWarning: variable 'volcello' has multiple fill values {1e+20, 1e+20}, decoding all values to NaN.
  new_vars[k] = decode_cf_variable(


saving to /projects/SOCCOM/graemem/projects/esm4_ppe/data/processed/correlation/pearson.intpppico-limfepico.bymonth.nc
[###########################             ] | 68% Completed | 128.25 s

/home/graemem/.conda/envs/climpred/lib/python3.11/site-packages/dask/array/numpy_compat.py:51: RuntimeWarning: invalid value encountered in divide
  x = np.divide(x1, x2, out)


... (15 outputs left out)

/home/graemem/.conda/envs/climpred/lib/python3.11/site-packages/xarray/conventions.py:448: SerializationWarning: variable 'thkcello' has multiple fill values {1e+20, 1e+20}, decoding all values to NaN.
  new_vars[k] = decode_cf_variable(
/home/graemem/.conda/envs/climpred/lib/python3.11/site-packages/xarray/conventions.py:448: SerializationWarning: variable 'volcello' has multiple fill values {1e+20, 1e+20}, decoding all values to NaN.
  new_vars[k] = decode_cf_variable(


saving to /projects/SOCCOM/graemem/projects/esm4_ppe/data/processed/correlation/pearson.intpppico-limppico.bymonth.nc
[###########################             ] | 69% Completed | 144.47 s

/home/graemem/.conda/envs/climpred/lib/python3.11/site-packages/dask/array/numpy_compat.py:51: RuntimeWarning: invalid value encountered in divide
  x = np.divide(x1, x2, out)


[########################################] | 100% Completed | 155.75 s
